# 02. 데이터 분할 (Stratified Split)

`01_EDA`가 만든 `index.csv`를 받아서 학습에 쓸 분할을 확정합니다.

**이 노트북의 산출물** (이후 `03`, `04`, `05`가 모두 이걸 읽습니다)

| 파일 | 내용 |
|---|---|
| `split.csv` | path, label, label_idx, split(train/val/test) |
| `label_map.json` | 클래스명 ↔ 정수 인덱스 (**전 노트북 공통**) |
| `class_weights.csv` | 불균형 대응용 가중치 |
| `norm_stats.json` | 데이터셋 채널 평균·표준편차 |

**순서가 중요합니다.** 중복 제거 → 분할 → 가중치 계산.
중복을 남긴 채 분할하면 같은 이미지가 train과 test에 동시에 들어가
(data leakage) 테스트 성능이 부풀려집니다.

## 1. 준비

In [ ]:
import json
import hashlib
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

OUT = Path("outputs/garbage")
df = pd.read_csv(OUT / "metrics" / "index.csv")

print(f"입력: {len(df):,}행 / {df['label'].nunique()}클래스")
print(df["label"].value_counts().to_string())

## 2. 중복 이미지 제거

파일 내용을 MD5로 해싱해서 **바이트 단위로 완전히 동일한** 이미지를 찾습니다.
(리사이즈·재압축된 유사 중복은 이 방법으로 안 잡히지만, 그건 별개 문제이고
완전 중복만 제거해도 leakage의 대부분은 막힙니다.)

In [ ]:
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

df["md5"] = [file_md5(p) for p in df["path"]]

dup_mask = df.duplicated(subset="md5", keep="first")
n_dup = int(dup_mask.sum())

print(f"완전 중복 파일: {n_dup}개")

if n_dup:
    # 중복이 클래스 경계를 넘나드는지 확인 — 넘나든다면 라벨 자체가 모순
    g = df[df.duplicated(subset="md5", keep=False)].groupby("md5")["label"].nunique()
    cross = int((g > 1).sum())
    print(f"  이 중 서로 다른 클래스에 걸친 중복: {cross}건")
    if cross:
        print("  → 동일 이미지가 두 클래스에 라벨링된 것이므로 반드시 제거 대상입니다.")
    print()
    print("제거되는 파일의 클래스 분포:")
    print(df.loc[dup_mask, "label"].value_counts().to_string())

df = df[~dup_mask].drop(columns="md5").reset_index(drop=True)
print(f"\n중복 제거 후: {len(df):,}행")

## 3. 라벨 인코딩 (전 노트북 공통 기준)

In [ ]:
classes = sorted(df["label"].unique())
label_map = {c: i for i, c in enumerate(classes)}

df["label_idx"] = df["label"].map(label_map)

with open(OUT / "metrics" / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print("클래스 인덱스")
for c, i in label_map.items():
    print(f"  {i:2d}  {c}")

> `label_map.json`을 파일로 고정하는 이유: `03`, `04`, `05`가 각자 `sorted()`를 다시 돌리면
> 클래스 순서가 어긋날 위험이 있고, 그러면 혼동행렬 축이 뒤바뀝니다.
> 이후 노트북은 이 파일을 **읽기만** 하세요.

## 4. Stratified Split (70 / 15 / 15)

`train_test_split`을 두 번 적용합니다. 두 번째 호출에서 `test_size=0.5`인 이유는
남은 30%를 val 15% / test 15%로 반씩 나누기 위해서입니다.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED, shuffle=True)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED, shuffle=True)

train_df = train_df.assign(split="train")
val_df   = val_df.assign(split="val")
test_df  = test_df.assign(split="test")

split_df = (pd.concat([train_df, val_df, test_df])
              .sort_values(["split", "label"])
              .reset_index(drop=True))

print(f"train {len(train_df):,}  |  val {len(val_df):,}  |  test {len(test_df):,}")
print(f"합계   {len(split_df):,}  (원본 {len(df):,})")
assert len(split_df) == len(df), "분할 과정에서 행 수가 바뀌었습니다"
assert split_df["path"].nunique() == len(split_df), "동일 경로가 중복 배정되었습니다"

## 5. 분할 검증 — 비율이 유지되었는가

In [ ]:
pivot = split_df.pivot_table(index="label", columns="split",
                             values="path", aggfunc="count")
pivot = pivot[["train", "val", "test"]]
pivot["total"] = pivot.sum(axis=1)

ratio = pivot[["train", "val", "test"]].div(pivot["total"], axis=0) * 100
ratio.columns = ["train%", "val%", "test%"]

print(pd.concat([pivot, ratio.round(1)], axis=1).to_string())
print()
print("전체 비율(%):",
      (pivot[["train", "val", "test"]].sum() / pivot["total"].sum() * 100).round(1).to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(pivot))
colors = {"train": "#4a7ba7", "val": "#e8a33d", "test": "#c0392b"}

for s in ["train", "val", "test"]:
    ax.bar(pivot.index, pivot[s], bottom=bottom, label=s, color=colors[s])
    bottom += pivot[s].values

ax.set_ylabel("images")
ax.set_title("Split composition per class")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUT / "figures" / "06_split.png", dpi=150)
plt.show()

**확인할 것**: 모든 클래스에서 train/val/test 비율이 70/15/15 근처여야 합니다.
가장 작은 클래스도 val·test에 최소 수십 장은 들어가야 클래스별 recall이 의미를 갖습니다.
어떤 클래스의 test 표본이 20장 미만이면 그 클래스의 recall은 통계적으로 불안정하니
보고서에서 그 점을 명시하세요.

## 6. 불균형 대응 가중치

두 가지 방식을 준비합니다. **어느 쪽이 나은지는 `03`, `04`에서 실험으로 비교**하고,
그 비교가 보고서의 핵심 근거가 됩니다.

- **class_weight**: 손실 함수에 넘김 (`nn.CrossEntropyLoss(weight=...)`).
  소수 클래스의 오답에 더 큰 페널티를 줍니다.
- **sample_weight**: `WeightedRandomSampler`에 넘김.
  배치를 구성할 때 소수 클래스를 더 자주 뽑습니다.

둘을 **동시에 쓰면 보정이 이중 적용**되어 소수 클래스로 과하게 쏠립니다. 하나만 쓰세요.

In [ ]:
train_counts = train_df["label"].value_counts().reindex(classes)
n_samples, n_classes = len(train_df), len(classes)

# sklearn 'balanced' 공식
class_weight = n_samples / (n_classes * train_counts)

cw = pd.DataFrame({
    "label":        classes,
    "label_idx":    [label_map[c] for c in classes],
    "train_count":  train_counts.values,
    "class_weight": class_weight.values.round(4),
}).sort_values("label_idx")

cw.to_csv(OUT / "metrics" / "class_weights.csv", index=False, encoding="utf-8")
print(cw.to_string(index=False))
print(f"\n가중치 범위: {class_weight.min():.3f} ~ {class_weight.max():.3f}")

In [ ]:
# 03/04 노트북에서 쓸 형태 — 참고용 스니펫
print("""
# ── (A) 손실 가중치 방식 ──────────────────────────────
cw = pd.read_csv("outputs/garbage/metrics/class_weights.csv").sort_values("label_idx")
w  = torch.tensor(cw["class_weight"].values, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=w)

# ── (B) 샘플러 방식 ───────────────────────────────────
from torch.utils.data import WeightedRandomSampler
per_class = cw.set_index("label_idx")["class_weight"].to_dict()
sample_w  = [per_class[i] for i in train_dataset.targets]
sampler   = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, ...)
# 주의: sampler를 쓰면 shuffle=True를 함께 줄 수 없습니다.
""")

## 7. 데이터셋 채널 통계 (mean / std)

정규화 상수를 **train split에서만** 계산합니다. val·test까지 포함해 계산하면
테스트 정보가 학습 전처리에 새어 들어갑니다.

- **직접 만든 CNN** (`03`) → 이 데이터셋 통계 사용
- **전이학습** (`04`) → ImageNet 통계 사용 (사전학습 가중치가 그 분포로 학습되었으므로)

In [ ]:
SIZE = 128          # 통계 추정용 축소 크기 — 224로 해도 값은 거의 동일
psum, psum_sq, npix = np.zeros(3), np.zeros(3), 0

for i, p in enumerate(train_df["path"]):
    img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is None:
        continue
    img = cv2.cvtColor(cv2.resize(img, (SIZE, SIZE)), cv2.COLOR_BGR2RGB)
    x = img.astype(np.float64) / 255.0
    psum    += x.sum(axis=(0, 1))
    psum_sq += (x ** 2).sum(axis=(0, 1))
    npix    += x.shape[0] * x.shape[1]
    if (i + 1) % 2000 == 0:
        print(f"  {i + 1:,} / {len(train_df):,}")

mean = psum / npix
std  = np.sqrt(psum_sq / npix - mean ** 2)

stats = {
    "dataset_mean": [round(float(v), 4) for v in mean],
    "dataset_std":  [round(float(v), 4) for v in std],
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std":  [0.229, 0.224, 0.225],
    "computed_on":   "train split only",
    "n_images":      int(len(train_df)),
}

with open(OUT / "metrics" / "norm_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print("\ndataset mean:", stats["dataset_mean"])
print("dataset std :", stats["dataset_std"])
print("imagenet    :", stats["imagenet_mean"], stats["imagenet_std"])

## 8. 저장 및 최종 점검

In [ ]:
split_df[["path", "label", "label_idx", "split"]].to_csv(
    OUT / "metrics" / "split.csv", index=False, encoding="utf-8")

# ── leakage 최종 확인 ──
sets = {s: set(split_df.loc[split_df.split == s, "path"]) for s in ["train", "val", "test"]}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    inter = sets[a] & sets[b]
    print(f"{a:5s} ∩ {b:5s} : {len(inter)}건", "OK" if not inter else "!!! 겹침 발생")

# ── 파일 실재 여부 표본 확인 ──
missing = [p for p in split_df["path"].sample(300, random_state=SEED) if not Path(p).exists()]
print(f"\n표본 300개 중 없는 파일: {len(missing)}개")

print(f"\nsplit.csv 저장 완료 — {len(split_df):,}행")
print(split_df.head(3).to_string(index=False))

---

## 다음 단계로 넘기는 것

```python
# 03, 04 노트북 공통 로딩부
import json, pandas as pd

split = pd.read_csv("outputs/garbage/metrics/split.csv")
label_map = json.load(open("outputs/garbage/metrics/label_map.json", encoding="utf-8"))
stats = json.load(open("outputs/garbage/metrics/norm_stats.json", encoding="utf-8"))

train_df = split[split.split == "train"]
val_df   = split[split.split == "val"]
test_df  = split[split.split == "test"]
```

`split.csv`가 경로+라벨만 담고 있으므로 **파일을 물리적으로 복사하지 않습니다.**
디스크 15,000장을 세 벌 만들 필요가 없고, 분할 기준을 바꾸고 싶으면 이 노트북만 다시 돌리면 됩니다.

→ `03P_03_baseline_cnn.ipynb` 로 이동